# Test crops: how good are the masks?

Runs a model on the 56 test-strip crops, 12 m across, with ground already removed. Each first round is 2 trunk clicks placed using the true labels, so this measures **mask quality given a good click**. It says nothing about automatic clicks, which `03_benchmark.ipynb` measures on whole tiles.

**Treat this as a final check.** Comparing models here is fine, but don't change settings because of these numbers, or the test set stops being a fair exam.

- `val_iou_pick`: IoU of the mask the model itself chooses from that first round, which is how real use chooses.
- `val_iou_first`: IoU of the best of those 3 masks, given the true answer.
- `val_iou`: IoU after correction clicks.

(The names say `val` because they come from the same function as validation.) Needs a GPU.

In [ ]:
import os
import sys

if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
sys.path.insert(0, "src")

import matplotlib.pyplot as plt
import pandas as pd
import torch
import yaml
from torch.utils.data import DataLoader

from treeseg.data.dataset import ForestCrops
from treeseg.engine.click_demo import predict_one_click
from treeseg.engine.trainer import collate, validate
from treeseg.models.point_sam import add_point_sam_to_path, build_point_sam
from treeseg.utils.view3d import plot_mask_candidates

cfg = yaml.safe_load(open(os.environ.get("TREESEG_CONFIG", "configs/experiment/benchmark.yaml")))
add_point_sam_to_path()
from pc_sam.model.loss import Criterion

print("device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NO GPU: this notebook will fail")
checkpoints = {name: path for name, path in cfg["checkpoints"].items() if os.path.exists(path)}
print("checkpoints found:", list(checkpoints))

test_set = ForestCrops("data/wytham_isprs/crops/test", 10000, 4, 40, seed=2)
loader = DataLoader(test_set, batch_size=1, shuffle=False, collate_fn=collate)
print("test crops:", len(test_set))

## Scores on all test crops

The same crops, clicks and sampled points for every model. Results are saved to `results/test_crops/summary.csv`.

In [ ]:
rows, examples = [], {}
for name, path in checkpoints.items():
    model = build_point_sam(path, cfg["group_number"], cfg["group_size"]).to("cuda")
    rows.append({"checkpoint": name, **validate(model, loader, Criterion(use_soft_iou=False), "cuda", "stem")})
    #the same few trees for each model, so the pictures can be compared
    examples[name] = [predict_one_click(model, test_set[index], 0, "stem1") for index in (5, 20, 40)]
    del model
    torch.cuda.empty_cache()

summary = pd.DataFrame(rows).round(3)
os.makedirs("results/test_crops", exist_ok=True)
summary.to_csv("results/test_crops/summary.csv", index=False)
summary

## See it

Three test trees. The first panel is the true tree, then the 3 masks for one trunk click, the green diamond. Compare models for the same crop.

In [ ]:
for number, index in enumerate((5, 20, 40)):
    for name in examples:
        result = examples[name][number]
        plot_mask_candidates(result, f"Test crop {index}: {name}")
        plt.savefig(f"results/test_crops/crop{index}_{name}.png")
        plt.show()